# Обучение сверточных моделей

Обучение двух сверточных моделей, одну - на полном датасете, вторую - на датасете без одного из классов.

## Импорты

In [15]:
import torch
import numpy as np
import kagglehub
import numpy.typing as npt
from torch import nn
from pathlib import Path
from tqdm import trange
from torch import optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from typing import override, Any, Literal
from pydantic import BaseModel

SEED: int = 7


def set_seed() -> None:
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    np.random.seed(SEED)


set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Структура модели

Модель представляет собой 3-х блочную CNN с классификатором, оптимизированную под изображения CIFAR10. Модель имеет компактный объем, что позволяет быстро проводить эксперименты.

In [2]:
class ConvolutionalModel(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features: nn.Module = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head: nn.Module = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    @override
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(input))

## Датасеты

In [3]:
default_transform = [
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
]

dataset_path = kagglehub.dataset_download("pankrzysiu/cifar10-python")

train_raw = datasets.CIFAR10(
    root=dataset_path,
    train=True,
    transform=transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            *default_transform,
        ]
    ),
)
test_raw = datasets.CIFAR10(
    root=dataset_path,
    train=False,
    transform=transforms.Compose(default_transform),
)

FORGET_CLASS = 0
train_raw.classes[FORGET_CLASS]

'airplane'

Для обучения и тестирования используется суммарно 6 датасетов. Пары тренировочных и тестовых датасетов:
- Весь датасет CIFAR10: $D_{\text{full}}$
- Один класс из $D_{\text{full}}$: $D_{\text{forget}}$
- $D_{\text{full}}$ без одного из классов: $D_{\text{retain}} = D_{\text{full}} \setminus D_{\text{forget}}$

In [4]:
train_targets = np.array(train_raw.targets)
test_targets = np.array(test_raw.targets)

train_retain_idx = np.where(train_targets != FORGET_CLASS)[0]
test_retain_idx = np.where(test_targets != FORGET_CLASS)[0]
train_forget_idx = np.where(train_targets == FORGET_CLASS)[0]
test_forget_idx = np.where(test_targets == FORGET_CLASS)[0]

BATCH_SIZE: int = 128

full_train_loader = DataLoader(
    train_raw, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
retain_train_loader = DataLoader(
    Subset(train_raw, train_retain_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)
forget_train_loader = DataLoader(
    Subset(train_raw, train_forget_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)

full_test_loader = DataLoader(test_raw, batch_size=BATCH_SIZE, shuffle=False)
retain_test_loader = DataLoader(
    Subset(test_raw, test_retain_idx.tolist()), batch_size=BATCH_SIZE * 2, shuffle=False
)
forget_test_loader = DataLoader(
    Subset(test_raw, test_forget_idx.tolist()), batch_size=BATCH_SIZE * 2, shuffle=False
)

## Обучение

### Метрики

Метрики разделены на три среза данных: $D_{\text{full}}$, $D_{\text{retain}}$, $D_{\text{forget}}$ по обучающим и тестовым выборкам:
- Среднее значение Cross-Entropy Loss
- Доля верных ответов
- Уверенность модели в забываемом классе: $\frac{1}{\vert{}D\vert{}} \sum_{x \in D} \text{Softmax}(f(x))_{c_{\text{forget}}}$. Для full-baseline модели эта метрика тоже считается. Для нее забываемый класс $c_{\text{forget}} = 0$
- Энтропия: $-\frac{1}{\vert{}D\vert{}}\sum_{x \in D}\sum_{}\text{Softmax}(f(x)) * ln(\text{Softmax}(f(x)))$
- Точность на каждом из классов


In [5]:
class EpochMetrics(BaseModel):
    # loss
    full_test_loss: float
    retain_test_loss: float
    forget_test_loss: float

    full_train_loss: float
    retain_train_loss: float
    forget_train_loss: float

    # accuracy
    full_test_accuracy: float
    retain_test_accuracy: float
    forget_test_accuracy: float

    full_train_accuracy: float
    retain_train_accuracy: float
    forget_train_accuracy: float

    # forget class probability
    full_test_forget_class_probability: float
    retain_test_forget_class_probability: float
    forget_test_forget_class_probability: float
    forget_test_confusion_targets: dict[int, float]

    full_train_forget_class_probability: float
    retain_train_forget_class_probability: float
    forget_train_forget_class_probability: float
    forget_train_confusion_targets: dict[int, float]

    # entropy
    full_test_entropy: float
    retain_test_entropy: float
    forget_test_entropy: float

    full_train_entropy: float
    retain_train_entropy: float
    forget_train_entropy: float

    # accuracy per class
    full_test_accuracy_per_class: dict[int, float]
    retain_test_accuracy_per_class: dict[int, float]
    forget_test_accuracy_per_class: dict[int, float]

    full_train_accuracy_per_class: dict[int, float]
    forget_train_accuracy_per_class: dict[int, float]
    retain_train_accuracy_per_class: dict[int, float]


def compute_epoch_metrics(model: nn.Module) -> EpochMetrics:
    model.eval()

    def evaluate_split(
        loader: DataLoader[Any],
    ) -> tuple[
        float, npt.NDArray[np.integer], npt.NDArray[np.integer], npt.NDArray[np.float64]
    ]:
        losses, predictions, targets, probabilities = [], [], [], []
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = nn.functional.cross_entropy(logits, y, reduction="none")
                probability = nn.functional.softmax(logits, dim=-1)

                losses.extend(loss.cpu().numpy())
                predictions.extend(logits.argmax(dim=-1).cpu().numpy())
                targets.extend(y.cpu().numpy())
                probabilities.append(probability.cpu().numpy())

        return (
            float(np.mean(np.array(losses))),
            np.array(predictions),
            np.array(targets),
            np.vstack(probabilities),
        )

    splits: dict[
        str,
        tuple[
            float,
            npt.NDArray[np.integer],
            npt.NDArray[np.integer],
            npt.NDArray[np.float64],
        ],
    ] = {
        "full_test": evaluate_split(full_test_loader),
        "retain_test": evaluate_split(retain_test_loader),
        "forget_test": evaluate_split(forget_test_loader),
        "full_train": evaluate_split(full_train_loader),
        "retain_train": evaluate_split(retain_train_loader),
        "forget_train": evaluate_split(forget_train_loader),
    }

    metrics: dict[str, float | dict[int, float]] = {}

    for split_name, (loss, predictions, targets, probabilities) in splits.items():
        metrics[f"{split_name}_loss"] = loss

        metrics[f"{split_name}_accuracy"] = float(np.mean(predictions == targets))
        metrics[f"{split_name}_forget_class_probability"] = float(
            np.mean(probabilities[:, FORGET_CLASS])
        )

        if split_name in ("forget_test", "forget_train"):
            unique, counts = np.unique(predictions, return_counts=True)
            total_forget = len(predictions)
            confusion_targets = {cls: 0.0 for cls in range(10)}
            for cls, count in zip(unique, counts):
                confusion_targets[int(cls)] = float(count / total_forget)

            metrics[f"{split_name}_confusion_targets"] = confusion_targets

        eps = 1e-12
        metrics[f"{split_name}_entropy"] = float(
            np.mean(
                -np.sum(
                    probabilities * np.log(probabilities + eps),
                    axis=1,
                )
            )
        )

        metrics[f"{split_name}_accuracy_per_class"] = {
            int(cls): float(np.mean(predictions[targets == cls] == cls))
            for cls in np.unique(targets)
        }

    return EpochMetrics.model_validate(metrics)

### Тренировка

In [6]:
def train_model(
    model: nn.Module, train_loader: DataLoader[Any], epochs: int = 15, lr: float = 1e-3
) -> list[EpochMetrics]:
    set_seed()
    metrics: list[EpochMetrics] = []
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    epoch_pbar = trange(1, epochs + 1, desc="Training")

    for _ in epoch_pbar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
        scheduler.step()
        metrics.append(compute_epoch_metrics(model))
        epoch_pbar.set_postfix(
            {
                "loss": f"{total_loss / total:.3f}",
                "acc": f"{correct * 100 / total:.1f}%",
            }
        )
    return metrics

In [11]:
class TrainingRunRecord(BaseModel):
    model_type: Literal["full_baseline", "retain_oracle", "unlearn"]
    forget_class: int | None
    epochs: int
    learning_rate: float
    batch_size: int
    metrics: list[EpochMetrics]

In [13]:
EPOCHS = 30
LEARNING_RATE = 1e-3

После обучения модели сохраняются [здесь](https://www.kaggle.com/models/ycalkk/convolutional-unlearning).

#### Full baseline

In [14]:
full_model = ConvolutionalModel()
full_model_metrics = train_model(
    full_model, full_train_loader, epochs=EPOCHS, lr=LEARNING_RATE
)
experiment_record = TrainingRunRecord(
    model_type="full_baseline",
    forget_class=None,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    metrics=full_model_metrics,
)

Training: 100%|██████████| 30/30 [17:23<00:00, 34.78s/it, loss=0.462, acc=83.9%]


In [16]:
save_dir = Path("./kaggle_export/full-baseline")
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(full_model.state_dict(), save_dir / "weights.pt")
with open(save_dir / "experiment_metadata.json", "w", encoding="utf-8") as f:
    f.write(experiment_record.model_dump_json(indent=2))
kagglehub.model_upload(
    handle="ycalkk/convolutional-unlearning/pyTorch/full-baseline",
    local_model_dir=str(save_dir),
    version_notes=f"Full baseline trained on CIFAR-10 ({EPOCHS} epochs)",
)

Uploading Model https://api.kaggle.com/models/ycalkk/convolutional-unlearning/pyTorch/full-baseline ...
Model 'convolutional-unlearning' does not exist or access is forbidden for user 'ycalkk'. Creating or handling Model...
Starting upload for file kaggle_export/full-baseline/experiment_metadata.json


Uploading: 100%|██████████| 85.5k/85.5k [00:00<00:00, 134kB/s]

Upload successful: kaggle_export/full-baseline/experiment_metadata.json (83KB)
Starting upload for file kaggle_export/full-baseline/weights.pt



Uploading: 100%|██████████| 1.44M/1.44M [00:00<00:00, 2.30MB/s]

Upload successful: kaggle_export/full-baseline/weights.pt (1MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/ycalkk/convolutional-unlearning/pyTorch/full-baseline


#### Retain oracle

In [18]:
retain_model = ConvolutionalModel()
retain_model_metrics = train_model(
    retain_model, retain_train_loader, epochs=EPOCHS, lr=LEARNING_RATE
)
experiment_record = TrainingRunRecord(
    model_type="retain_oracle",
    forget_class=FORGET_CLASS,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    metrics=retain_model_metrics,
)

Training: 100%|██████████| 30/30 [16:53<00:00, 33.78s/it, loss=0.411, acc=85.5%]


In [19]:
save_dir = Path(f"./kaggle_export/retain-oracle-c{FORGET_CLASS}")
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(retain_model.state_dict(), save_dir / "weights.pt")
with open(save_dir / "experiment_metadata.json", "w", encoding="utf-8") as f:
    f.write(experiment_record.model_dump_json(indent=2))
kagglehub.model_upload(
    handle=f"ycalkk/convolutional-unlearning/pyTorch/retain-oracle-c{FORGET_CLASS}",
    local_model_dir=str(save_dir),
    version_notes=f"Retain oracle without {FORGET_CLASS} class trained on CIFAR-10 ({EPOCHS} epochs)",
)

Uploading Model https://api.kaggle.com/models/ycalkk/convolutional-unlearning/pyTorch/retain-oracle-c0 ...
Starting upload for file kaggle_export/retain-oracle-c0/experiment_metadata.json


Uploading: 100%|██████████| 85.2k/85.2k [00:00<00:00, 139kB/s]

Upload successful: kaggle_export/retain-oracle-c0/experiment_metadata.json (83KB)
Starting upload for file kaggle_export/retain-oracle-c0/weights.pt



Uploading: 100%|██████████| 1.44M/1.44M [00:00<00:00, 2.32MB/s]

Upload successful: kaggle_export/retain-oracle-c0/weights.pt (1MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/ycalkk/convolutional-unlearning/pyTorch/retain-oracle-c0
